In [2]:
import numpy as np
import math
import sys
sys.path.append("../")
from bareMC.noise_generator import ColoredNoiseGenerator_FourierFiltering
from bareMC.NMSSE_bare_MC import NMSSE_Linear_Bare_MC
from bareMC.exp_val import compute_exp_val

In [3]:
# Spin-Boson model parameters
Delta = 1
eps = 0
sx = np.array([[0, 1], [1, 0]], dtype=complex)
sz = np.array([[1, 0], [0, -1]], dtype=complex)
Hs = -0.5*Delta*sx + 0.5*eps*sz
L = sz
# bath_corr = lambda t: 2*np.exp(-(0.5+2j)*abs(t))
tmax = 5/Delta

## Comparison with Cai_2020_CPAM

In [ ]:
# parameters in Cai_2020_CPAM
beta = 5/Delta
wc = 2.5 * Delta
wmax = 4 * wc
CapL = 200
factor = 1 - np.exp(-wmax/wc)
wl = -wc * np.log(1 - np.linspace(1, CapL + 1, CapL)/CapL * factor)
cl = wl * np.sqrt((0.2 * wc/CapL) * factor)

_coth = 1.0 / np.tanh(0.5 * beta * wl)
_pref = (cl**2) / (2.0 * wl)
def bath_corr(t):
    """Bath correlation function alpha(t).
    Supports scalar t or 1D array t (returns same shape).
    """
    t_arr = np.asarray(t)
    t1 = np.atleast_1d(t_arr).astype(float)
    phase = wl[:, None] * t1[None, :]             # shape: (CapL, len(t))
    out = np.sum(_pref[:, None] * (np.cos(phase) * _coth[:, None] - 1j * np.sin(phase)), axis=0)
    return out[0] if t_arr.ndim == 0 else out

# NMSSE parameters
solver = NMSSE_Linear_Bare_MC(
    Hs = Hs,
    L = L,
    tmax = tmax,
    bath_corr = bath_corr,
    N_trunc = 6,
    N_grid = 100
)
psis = solver.compute_realization(N_traj=100)
sigma_z = compute_exp_val(psis, sz)
print(sigma_z)

## Comparison with HOPS

### N_traj = 20 and N_grid = 100

In [5]:
tlist = np.array([2, 5, 10, 15]) / Delta
def alpha(t):
    """
    Bath correlation function alpha(t).
    Supports scalar t or array-like t (returns matching shape).
    """
    t_arr = np.asarray(t)
    out = 2 * np.exp(-(0.5 + 2j) * np.abs(t_arr))
    return out.item() if t_arr.ndim == 0 else out

for tmax in tlist:
    solver_hops = NMSSE_Linear_Bare_MC(
        Hs = Hs,
        L = L,
        tmax = tmax,
        bath_corr = alpha,
        N_trunc = 6,
        N_grid = 100
    )
    psis_hops = solver_hops.compute_realization(N_traj=20)
    # Quick diagnostics: check if the state norm collapses (underflow) or explodes
    norms = np.linalg.norm(psis_hops, axis=1)
    print(f"t*Delta={tmax*Delta}: min||psi||={norms.min():.3e}, max||psi||={norms.max():.3e}, max|psi|={np.max(np.abs(psis_hops)):.3e}")
    sigma_z_hops = compute_exp_val(psis_hops, sz)
    print(f"t*Delta = {tmax*Delta}, <sigma_z> = {sigma_z_hops}")

t*Delta=2.0: min||psi||=3.181e-01, max||psi||=1.880e+01, max|psi|=1.873e+01
t*Delta = 2.0, <sigma_z> = 0.8919023537782675
t*Delta=5.0: min||psi||=2.084e+01, max||psi||=1.831e+02, max|psi|=1.644e+02
t*Delta = 5.0, <sigma_z> = -0.15411970998794983
t*Delta=10.0: min||psi||=4.725e+02, max||psi||=1.592e+04, max|psi|=1.247e+04
t*Delta = 10.0, <sigma_z> = -0.24101456484450254
t*Delta=15.0: min||psi||=3.405e+03, max||psi||=2.115e+04, max|psi|=1.806e+04
t*Delta = 15.0, <sigma_z> = 0.11500478698622503


### N_traj = 100 and N_grid = t * 50 

In [ ]:
tlist = np.array([2, 5]) / Delta

for tmax in tlist:
    solver_hops = NMSSE_Linear_Bare_MC(
        Hs = Hs,
        L = L,
        tmax = tmax,
        bath_corr = alpha,
        N_trunc = 6,
        N_grid = int(tmax * 50)
    )
    psis_hops = solver_hops.compute_realization(N_traj=100)
    sigma_z_hops = compute_exp_val(psis_hops, sz)
    print(f"t*Delta = {tmax*Delta}, <sigma_z> = {sigma_z_hops}")

t*Delta = 2.0, <sigma_z> = 0.8512722553082369
t*Delta = 5.0, <sigma_z> = 0.1598988783357542


In [ ]:
tlist = np.array([10, 12]) / Delta

for tmax in tlist:
    solver_hops = NMSSE_Linear_Bare_MC(
        Hs = Hs,
        L = L,
        tmax = tmax,
        bath_corr = alpha,
        N_trunc = 6,
        N_grid = int(tmax * 50)
    )
    psis_hops = solver_hops.compute_realization(N_traj=100)
    sigma_z_hops = compute_exp_val(psis_hops, sz)
    print(f"t*Delta = {tmax*Delta}, <sigma_z> = {sigma_z_hops}")

t*Delta = 10.0, <sigma_z> = -0.20863377798127303
t*Delta = 12.0, <sigma_z> = 0.029861324032142346


In [9]:
tlist = np.array([10]) / Delta

for tmax in tlist:
    solver_hops = NMSSE_Linear_Bare_MC(
        Hs = Hs,
        L = L,
        tmax = tmax,
        bath_corr = alpha,
        N_trunc = 6,
        N_grid = int(tmax * 50)
    )
    psis_hops = solver_hops.compute_realization(N_traj=100)
    sigma_z_hops = compute_exp_val(psis_hops, sz)
    print(f"t*Delta = {tmax*Delta}, <sigma_z> = {sigma_z_hops}")

t*Delta = 10.0, <sigma_z> = 0.310957121156922


In [11]:
tlist = np.array([10]) / Delta

for tmax in tlist:
    solver_hops = NMSSE_Linear_Bare_MC(
        Hs = Hs,
        L = L,
        tmax = tmax,
        bath_corr = alpha,
        N_trunc = 6,
        N_grid = int(tmax * 50)
    )
    psis_hops = solver_hops.compute_realization(N_MC_samples=1000, N_traj=300)
    sigma_z_hops = compute_exp_val(psis_hops, sz)
    print(f"t*Delta = {tmax*Delta}, <sigma_z> = {sigma_z_hops}")

t*Delta = 10.0, <sigma_z> = 0.09085183271978356


In [12]:
solver_hops = NMSSE_Linear_Bare_MC(
    Hs = Hs,
    L = L,
    tmax = 10 / Delta,
    bath_corr = alpha,
    N_trunc = 6,
    N_grid = int((10 / Delta) * 50)
)
psis_hops = solver_hops.compute_realization(N_MC_samples=1000, N_traj=300)
sigma_z_hops = compute_exp_val(psis_hops, sz)
print(f"t*Delta = {10}, <sigma_z> = {sigma_z_hops}")

t*Delta = 10, <sigma_z> = 0.3236158885390862


In [14]:
solver_hops = NMSSE_Linear_Bare_MC(
    Hs = Hs,
    L = L,
    tmax = 10 / Delta,
    bath_corr = alpha,
    N_trunc = 6,
    N_grid = int((10 / Delta) * 50)
)
psis_hops = solver_hops.compute_realization(N_MC_samples=1000, N_traj=300)
sigma_z_hops = compute_exp_val(psis_hops, sz)
print(f"t*Delta = {10}, <sigma_z> = {sigma_z_hops}")

t*Delta = 10, <sigma_z> = 0.1496276904456774
